# 29 — LFU Cache (Amazon FAR style)

Evict the **least-frequently-used** item (ties broken least-recently-used) — the O(1) LFU design with
frequency buckets. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch tasks
(frequency aging, then parallel hit-rate simulation). Fill stubs, run each test cell, peek at the
collapsed `ref_` solutions only after trying.

Internals: `key_freq[key]`, `freq_keys[f]` = an `OrderedDict` of keys at frequency `f` (LRU order),
and `min_freq`.

---

## Part 1 — LFU cache

`LFUCache(capacity)` with `get(key)` (returns value or `None`, and **increments** the key's frequency)
and `put(key, value)` (insert/update; on overflow evict the least-frequently-used, breaking ties by
least-recently-used).

**Lock down:** O(1) via frequency buckets + a `min_freq` pointer; eviction pops the LRU end of the
`min_freq` bucket; `get` and `put` both bump frequency.

In [ ]:
from collections import defaultdict, OrderedDict


class LFUCache:
    def __init__(self, capacity):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    c = LFUCache(2)
    c.put("a", 1); c.put("b", 2)
    assert c.get("a") == 1                              # a freq 2, b freq 1
    c.put("c", 3)                                       # evict LFU = b
    assert c.get("b") is None and c.get("a") == 1 and c.get("c") == 3
    d = LFUCache(2)
    d.put("a", 1); d.put("b", 2)                        # tie at freq 1; a is LRU
    d.put("c", 3)                                       # tie -> evict LRU = a
    assert d.get("a") is None and d.get("b") == 2 and d.get("c") == 3
    print("Part 1 OK")

_t1()

## Part 2 — Compute-on-miss

`get_or_compute(cache, key, fn)`: return the cached value, or compute it with `fn()`, store it, and
return it (memoization). Works on any `LFUCache`.

**Lock down:** one `get`, and on miss one `fn()` + `put`; assumes values are non-`None` (use a sentinel
if `None` is a valid value).

In [ ]:
def get_or_compute(cache, key, fn):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    c = LFUCache(2)
    calls = [0]

    def fn():
        calls[0] += 1
        return 99

    assert get_or_compute(c, "k", fn) == 99 and calls[0] == 1
    assert get_or_compute(c, "k", fn) == 99 and calls[0] == 1   # cached -> fn not called again
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe LFU

`ConcurrentLFU(capacity)`: thread-safe `get`/`put`. The frequency-bucket bookkeeping is delicate, so
guard every mutation. Many threads must `get`/`put` concurrently without corrupting the buckets or
`min_freq`.

**The invariant:** 8 threads each `put` 50 distinct keys into a capacity-50 cache leaves exactly 50
entries. **Discuss:** the whole get/put must be atomic (the `min_freq` update + bucket moves are
multi-step); striping by key for less contention.

In [ ]:
import threading


class ConcurrentLFU:
    def __init__(self, capacity):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

    def size(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    c = ConcurrentLFU(50)

    def worker(t):
        for i in range(50):
            c.put("t%d_%d" % (t, i), i)

    ts = [threading.Thread(target=worker, args=(t,)) for t in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert c.size() == 50                              # capacity respected, buckets not corrupted
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Frequency aging

A pure LFU never forgets: a once-popular item can squat forever (cache pollution). `decay(cache)`
**halves** every key's frequency (floor, minimum 1) and rebuilds the buckets, so stale-popular items
can eventually be evicted. Also handle `capacity <= 0` (a `put` is a no-op).

**Discuss:** aging / windowed LFU (LFU-DA, TinyLFU), the pollution problem, and why a plain
hit-frequency policy degrades when popularity shifts over time.

In [ ]:
def decay(cache):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    c = LFUCache(3)
    c.put("a", 1)
    for _ in range(10):
        c.get("a")                                     # a frequency -> 11
    c.put("b", 2)                                       # b frequency 1
    decay(c)                                            # a -> 5, b -> 1
    assert c.key_freq["a"] == 5 and c.key_freq["b"] == 1
    z = LFUCache(0); z.put("x", 1); assert z.get("x") is None   # capacity 0
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel hit-rate simulation

`parallel_hit_rates(logs, capacity, num_procs) -> list[int]`: replay many independent access traces
through an LFU of the given capacity, in parallel with `ProcessPoolExecutor`, returning each trace's
hit count. Worker is `lfu_workers.simulate` (miss inserts `key→key`).

**Discuss:** each trace is an independent simulation (parallel); this is how you sweep capacities /
policies over recorded traffic to tune a cache; GIL → processes for the CPU-bound replays.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import lfu_workers


def parallel_hit_rates(logs, capacity, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    log1 = ["a", "b", "a", "a", "c", "a"]              # cap 2: hits at a,a (after first),a -> 3
    log2 = ["x", "x", "x"]                              # 2 hits
    assert parallel_hit_rates([log1, log2], capacity=2, num_procs=2) == [3, 2]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- LFU vs LRU vs ARC vs W-TinyLFU; when frequency beats recency and vice versa.
- Exact O(1) LFU (this) vs approximate (sketch-based counters) for huge key spaces.
- Sharded LFU for throughput; admission policies (TinyLFU's frequency filter).

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import defaultdict, OrderedDict
import threading
from concurrent.futures import ProcessPoolExecutor
import lfu_workers


class RefLFUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.key_val = {}
        self.key_freq = {}
        self.freq_keys = defaultdict(OrderedDict)       # freq -> OrderedDict of keys (LRU order)
        self.min_freq = 0

    def _touch(self, key):
        f = self.key_freq[key]
        del self.freq_keys[f][key]
        if not self.freq_keys[f]:
            del self.freq_keys[f]
            if self.min_freq == f:
                self.min_freq = f + 1
        self.key_freq[key] = f + 1
        self.freq_keys[f + 1][key] = None

    def get(self, key):
        if key not in self.key_val:
            return None
        self._touch(key)
        return self.key_val[key]

    def put(self, key, value):
        if self.cap <= 0:
            return
        if key in self.key_val:
            self.key_val[key] = value
            self._touch(key)
            return
        if len(self.key_val) >= self.cap:
            ek, _ = self.freq_keys[self.min_freq].popitem(last=False)   # LFU, LRU tie-break
            del self.key_val[ek]
            del self.key_freq[ek]
        self.key_val[key] = value
        self.key_freq[key] = 1
        self.freq_keys[1][key] = None
        self.min_freq = 1


def ref_get_or_compute(cache, key, fn):
    v = cache.get(key)
    if v is not None:
        return v
    v = fn()
    cache.put(key, v)
    return v


class RefConcurrentLFU(RefLFUCache):
    def __init__(self, capacity):
        super().__init__(capacity)
        self.lock = threading.Lock()

    def get(self, key):
        with self.lock:
            return super().get(key)

    def put(self, key, value):
        with self.lock:
            super().put(key, value)

    def size(self):
        with self.lock:
            return len(self.key_val)


def ref_decay(cache):
    new_freq, new_buckets = {}, defaultdict(OrderedDict)
    for key in cache.key_val:
        f = max(1, cache.key_freq[key] // 2)
        new_freq[key] = f
        new_buckets[f][key] = None
    cache.key_freq = new_freq
    cache.freq_keys = new_buckets
    cache.min_freq = min(new_buckets) if new_buckets else 0


def ref_parallel_hit_rates(logs, capacity, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(lfu_workers.simulate, [(capacity, log) for log in logs]))


_c = RefLFUCache(2); _c.put("a", 1); _c.put("b", 2); _c.get("a"); _c.put("c", 3)
assert _c.get("b") is None and _c.get("a") == 1
_m = RefLFUCache(1); _calls = [0]
assert ref_get_or_compute(_m, "k", lambda: (_calls.__setitem__(0, _calls[0] + 1), 7)[1]) == 7
assert ref_get_or_compute(_m, "k", lambda: 0) == 7 and _calls[0] == 1
_d = RefLFUCache(2); _d.put("a", 1)
for _ in range(7):
    _d.get("a")
ref_decay(_d); assert _d.key_freq["a"] == 4                # freq 8 -> 4
assert ref_parallel_hit_rates([["a", "a", "a"]], 1, 2) == [2]
print("reference solutions OK")